# Building Data Apps with Streamlit and Copilot

Keep this notebook open throughout the workshop - it contains the exercises you'll complete as we go, plus code snippets from the slides.


## Module 1: Introduction & Setup

In this section you'll start by loading and displaying a dataset in the notebook — this is the same dataset we'll use throughout the course. Then you'll port that code to a Streamlit app.

### Exercise 1.1: Jupyter

The code below loads and displays the dataset we'll be using throughout the course. Click the cell to select it, then click the "Run" button in the toolbar above (or press Shift+Enter) to run it. What can you learn about the dataset from the output?

*Bonus*: Which state had the lowest population?

In [ ]:
import pandas as pd

df = pd.read_csv("state_data.csv")

df

### Exercise 1.2: Streamlit

You'll start by running "The World's Smallest Streamlit App", and then update it to display the same dataset from the exercise above.

1. Open the file `1-intro.py` using the code editor of your choice. This file simply imports the `streamlit` module. Can you guess what will happen when you run it?
2. Run the app by doing the following:
    1. Open a new terminal
    2. Go to the course directory
    3. Activate the course's virtual environment ([instructions](https://github.com/arilamstein/streamlit-workshop/blob/main/SETUP.md#step-3-activate-the-virtual-environment))
    4. Run the app by typing `streamlit run 1-intro.py` in the console. 

A browser with a blank screen should appear. Now update the app to display the course dataset:

1. Replace the contents of `1-intro.py` with the code below:

In [ ]:
import streamlit as st
import pandas as pd

df = pd.read_csv("state_data.csv")

st.dataframe(df)

2. Save the file.
3. After saving the file, the word "Rerun" should appear in the app. Click it.
4. The app should update with a table.

*Bonus*: Which state had the lowest population?

## Module 2: User Input

### Dataframes: a Collection of Series

Here's the data frame we've been working with. Note that it is of type `DataFrame`.

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv("state_data.csv")
display(df)
display(type(df))

A DataFrame is a collection of columns, each of the same length. Each column has a name and a type (string, integer, etc.).

To work with a single column, type `df[<column name>]`.

The formal type of a column is `Series`.

In [ ]:
display(df["State"])
display(type(df["State"]))

The `Series` class has a lot of methods. The `unique()` method returns the unique values in the series:

In [ ]:
df["State"].unique()

Use `df["State"].unique()` to populate the options in the selectbox in `input_app.py`.

### Filtering a Dataframe (Boolean Indexing)

The most common operation we perform on a dataframe is: "Show me the rows where some condition is True". 

An example is "Show me the rows where the `State` column is `Wyoming`". 

Pandas does this with a technique called Boolean Indexing, which has two steps:
  1. Create a Boolean Series — a column of True/False values, one per row.
  2. Use that Series inside brackets to return only the rows where the value is True.

#### Step 1: Create the Boolean Series

To check which rows have `State == "Wyoming"`, type:

```py
df["State"] == "Wyoming"
```

This comparison is applied to every element in the column - a behavior called **vectorization**.

In [ ]:
# Display the column
display(df["State"])

# Vectorized comparison
df["State"] == "Wyoming"

#### Step 2: Use the Boolean Series to Filter

If you place that Boolean Series inside `df[...]`, pandas returns only the rows where the value is True:

In [ ]:
df[df["State"] == "Wyoming"]

That’s the entire pattern.
You’ll use this same structure to filter the dataframe based on the state the user selects in your Streamlit app.

### Exercise: Update the Selectbox

Using the code above, make the selectbox in `2-input.py` work as expected:

1. Use `df["State"].unique()` to populate the selectbox options.
2. Filter the dataframe to show only the rows for the selected state.

## Module 3: Graphics

We want to graph the population trends of a single state over time.

To do this, let's start by reading in our data and filtering it to a single state

In [ ]:
import pandas as pd

df = pd.read_csv("state_data.csv")

ca_mask = df["State"] == "California"
df_ca = df[ca_mask]
df_ca

### Plotly

In this course we'll be using the Plotly library to make interactive graphs. 

Plotly's "Plotly Express" module makes it easy to create graphs. Most functions have descriptive names (like `line` for making line graphs) that take:

  * A dataframe as the first agument
  * `x`: The column to use for the x-axis
  * `y`: The column to use for the y-axis
  * `title`: The text to use for the title

Here's an example that creates a line graph of the population of California over time. Note that when you hover over the line, you can see the exact values (year, population) of the point.

In [ ]:
import plotly.express as px

px.line(
    df_ca, x="Year", y="Total Population", title="Population of California over Time"
)

### Plotly Graphs in Streamlit

To output a plotly graph in a Streamlit app you must call the function `st.plotly_chart`.

A common pattern is:

```py
fig = px.line(
    df_ca, x="Year", y="Total Population", title="Population of California over Time"
)
st.plotly_chart(fig)
```

That is, it's common to store the result of `px.line` in a variable called `fig`. And then pass `fig` to `st.plotly_chart()`.

### f-Strings

F-strings let you embed variables inside curly braces `{}` for fast, readable string formatting. For example, here's how to use f-strings to create a title for a graph:

In [ ]:
state = "New York"
title = f"Graph for {state}"
title

### Exercise: Plotly Graphs in Streamlit

Update `3-graphics.py` to create a line graph of the population of the selected State. Use an f-string for the title of the graph. It should say the name of the state.

### Choosing What to Graph

Recall the structure of our data: there is one column for population and one column for income:

In [ ]:
df.head()

Up until now we've only graphed the population data. We can the income data by setting `y="Median Household Income"`. Run the code below to see.

In [ ]:
px.line(
    df_ca,
    x="Year",
    y="Median Household Income",
    title="Median Household Income of California over Time",
)

### Exercise: Selecting What to Graph

Update the app so that the user can select which column to graph. 

Here are the steps:
  1. Add a new select box to `3-graphics.py`. Populate it with the values "Total Population" and "Median Household Income". (Hint: pass those values as a list to `st.selectbox` i.e. `['Total Population', 'Median Household Income']`).
  2. Store the value returned from that selectbox in a variable called `demographic`.
  3. Use the value of `demographic` as the value for `y` in your line graph.

### Choropleth Map

A choropleth map shows regions (like states), and expresses values for those regions (like population) using color. 

Use `px.choropleth` to create a choropleth. Instead of specifying `x` and `y`, you specify:
  * `locations`: The column that identifies the location of the observation (ex. "New York").
  * `color`: The column you want to map to color (ex. "Total Population").

The `title` parameter is the same. But there are two other parameters unique to maps:
  * `scope='usa'` zooms the map in on the US.
  * `locationmode='usa-states'` clarifies that the location is recorded using two-letter state abbreviations to identify US states. This function does not understand full state names. So the location column must be `State Abbrev`.

Below is code to create a choropleth map of the population of US States for 2013. 

In [ ]:
df

In [ ]:
mask = df["Year"] == 2023
df_2023 = df[mask]

px.choropleth(
    df_2023,
    locations="State Abbrev",  # Column for region
    locationmode="USA-states",
    color="Total Population",  # Column for color
    scope="usa",
    title="2023 Total Population",
    color_continuous_scale="viridis",
)

### Exercise: Choropleth Map

1. Copy the above code, verbatim, to the app. Verify that it works.
2. Connect the map to the `demographic` selectbox. Do the results surprise you?
3. Create a new selectbox that lets the user select which `Year` of data to map.
4. Connect the year selectbox to the map. 

The result should be an app that lets users select which year and demographic statistic to map.

**Question**: How does the Total Population map change over time? How about the Median Household Income map?


## Module 4: User Interface (UI)

Note that you cannot run this code in a Jupyter Notebook: Streamlit builds websites that run in a browser, not a notebook. But you can copy it into `4-ui.py` and run it there by typing `streamlit run 4-ui.py` in the terminal.

### Columns

Create columns with `st.columns()`. It takes an integer (the number of columns you want) and returns a list of column objects as a list. It is common to use **iterable unpacking** to immediate store those column objects as variables.

In [ ]:
import streamlit as st

col1, col2 = st.columns(2)

You can use those column with a **context manager**. This means typing `with col1:` and then using indentation to indicate the output you want to see in the column:

In [ ]:
col1, col2 = st.columns(2)
with col1:
    st.write("I'm in col1...")
with col2:
    st.write("...and I'm in col2!")

Output above and below the context manager will use Streamlit's default of 1 column:

In [ ]:
st.title("Demo Streamlit App")

col1, col2 = st.columns(2)
with col1:
    st.write("I'm in col1...")
with col2:
    st.write("...and I'm in col2!")

st.write("Back to normal")

### Exercise: Columns

Open `4-ui.py` and run it with `streamlit run 4-ui.py`. 

The app has 3 select boxes at the top: state, demographic and year. Currently they appear below each other.

Update the app to have 3 columns, and put one select box in each column.

### Tabs

Create tabs with `st.tabs()`. It takes a **list** of **names** for each tab. Users will click the name of the tab to show what's inside it. It is common to use **iterable unpacking** to immediate store those tab objects as variables.

Because tabs have names, it is common to give the variables descriptive names. (Unlike columns, which are normally just called `col1`, `col2`, etc.)


In [ ]:
graph_tab, table_tab = st.tabs(["graph", "table"])

In [ ]:
with graph_tab:
    st.write("A graph should go here. Use st.plotly_chart()")
with table_tab:
    st.write("A table here please. Use st.dataframe()")

### Exercise: Tabs

Open `4-ui.py` and run it with `streamlit run 4-ui.py`. 

The app has 3 visualizations at the bottom: a line graph, a map, and a table. Currently they appear below each other.

Update the app to have 3 tabs, and put one visualization in each tab. 

Extra credit: ask an LLM to choose an emoji for each tab.

## 5. Deploying a Streamlit App

1. Create a repo in github. Fork existing course repo.

2. Create an account on Streamlit Cloud, using your github account.

3. Click "Create app", and follow the instructions.